In [1]:
!unzip Dataset-Movies.zip

Archive:  Dataset-Movies.zip
  inflating: dataset_train.csv       
  inflating: __MACOSX/._dataset_train.csv  
  inflating: dataset_test.csv        
  inflating: __MACOSX/._dataset_test.csv  


In [2]:
import random
import zipfile

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    precision_recall_fscore_support,
    f1_score,
    classification_report
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline, FeatureUnion

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

In [4]:
train_df = pd.read_csv("dataset_train.csv")
test_df = pd.read_csv("dataset_test.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)

display(train_df.head())
display(test_df.head())

Train: (8475, 3)
Test: (942, 2)


,movie_name,genre,description
0,Silent Hill,"Horror, Mystery","Rose, a desperate mother takes her adopted dau..."
1,Breaking the Waves,"Drama, Romance","In a small and conservative Scottish village, ..."
2,Wind Chill,"Drama, Horror, Thriller",Two college students share a ride home for the...
3,Godmothered,"Family, Fantasy, Comedy",A young and unskilled fairy godmother that ven...
4,Donkey Skin,"Fantasy, Comedy, Music, Romance",A fairy godmother helps a princess disguise he...


,movie_name,description
0,Opposites Attract,"She's a divorce lawyer, single mother and perp..."
1,A Turtle's Tale: Sammy's Adventures,A sea turtle who was hatched in 1959 spends th...
2,My Stepmother Is an Alien,Trying to rescue her home planet from destruct...
3,You've Got Mail,"Book superstore magnate, Joe Fox and independe..."
4,The Thing,"In the winter of 1982, a twelve-man research t..."


In [5]:
train_df["genre_list"] = train_df["genre"].apply(
    lambda x: [g.strip() for g in x.split(",")]
)

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(train_df["genre_list"])

genres = list(mlb.classes_)

print("Géneros:", genres)
print("Y shape:", Y.shape)

Géneros: ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']
Y shape: (8475, 18)


In [6]:
train_df["text"] = (
    "Title: " + train_df["movie_name"].fillna("") +
    ". Plot: " + train_df["description"].fillna("")
)

test_df["text"] = (
    "Title: " + test_df["movie_name"].fillna("") +
    ". Plot: " + test_df["description"].fillna("")
)

display(train_df[["movie_name", "genre", "text"]].head())

,movie_name,genre,text
0,Silent Hill,"Horror, Mystery","Title: Silent Hill. Plot: Rose, a desperate mo..."
1,Breaking the Waves,"Drama, Romance",Title: Breaking the Waves. Plot: In a small an...
2,Wind Chill,"Drama, Horror, Thriller",Title: Wind Chill. Plot: Two college students ...
3,Godmothered,"Family, Fantasy, Comedy",Title: Godmothered. Plot: A young and unskille...
4,Donkey Skin,"Fantasy, Comedy, Music, Romance",Title: Donkey Skin. Plot: A fairy godmother he...


In [7]:
X = train_df["text"].tolist()
X_test = test_df["text"].tolist()

X_train, X_dev, y_train, y_dev = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

print("X_train:", len(X_train))
print("X_dev:", len(X_dev))
print("X_test:", len(X_test))
print("y_train:", y_train.shape)
print("y_dev:", y_dev.shape)

X_train: 6780
X_dev: 1695
X_test: 942
y_train: (6780, 18)
y_dev: (1695, 18)


In [8]:
def compute_metrics_official(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)
    hl = hamming_loss(y_true, y_pred)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl
    }


def compute_metrics_debug(y_true, y_pred):
    metrics = compute_metrics_official(y_true, y_pred)
    metrics["avg_labels_pred"] = y_pred.sum(axis=1).mean()
    metrics["avg_labels_true"] = y_true.sum(axis=1).mean()
    return metrics

In [9]:
def find_best_thresholds(y_true, y_scores, genres, thresholds=np.arange(0.05, 0.96, 0.01)):
    best_thresholds = []
    best_f1s = []

    for j, genre in enumerate(genres):
        best_t = 0.5
        best_f1 = 0.0

        for t in thresholds:
            y_pred_j = (y_scores[:, j] >= t).astype(int)
            f1 = f1_score(y_true[:, j], y_pred_j, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        best_thresholds.append(best_t)
        best_f1s.append(best_f1)

    return np.array(best_thresholds), np.array(best_f1s)

In [10]:
MODEL_NAME = "roberta-base"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [11]:
class MovieGenreDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [12]:
train_dataset = MovieGenreDataset(X_train, y_train, tokenizer, MAX_LEN)
dev_dataset = MovieGenreDataset(X_dev, y_dev, tokenizer, MAX_LEN)
test_dataset = MovieGenreDataset(
    X_test,
    np.zeros((len(X_test), len(genres))),
    tokenizer,
    MAX_LEN
)

print(len(train_dataset), len(dev_dataset), len(test_dataset))

6780 1695 942


In [13]:
model_roberta = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(genres),
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
def compute_metrics_trainer(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    return compute_metrics_official(labels, preds)

In [17]:
training_args = TrainingArguments(
    output_dir="./roberta_base_ensemble_results",
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

In [18]:
print(training_args.device)

mps


In [19]:
trainer_roberta = Trainer(
    model=model_roberta,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics_trainer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print(trainer_roberta.args.device)

trainer_roberta.train()

mps


/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.295921,0.251046,0.141003,0.388988,0.567760,0.325621,0.104228,20.101600,84.322000,5.273000
2,0.220410,0.223736,0.194690,0.505822,0.650272,0.449204,0.092658,19.922500,85.079000,5.321000
3,0.187046,0.217026,0.204720,0.559998,0.652971,0.504400,0.088922,20.255200,83.682000,5.233000
4,0.165814,0.215025,0.221239,0.564255,0.648976,0.509370,0.087119,19.965800,84.895000,5.309000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3392, training_loss=0.21729759000382334, metrics={'train_runtime': 1213.7423, 'train_samples_per_second': 22.344, 'train_steps_per_second': 2.795, 'total_flos': 3568298450042880.0, 'train_loss': 0.21729759000382334, 'epoch': 4.0})

In [20]:
eval_roberta = trainer_roberta.evaluate()
display(eval_roberta)

/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
0.165814,0.215025,4,0.221239,0.564255,0.648976,0.509370,0.087119,20.569200,82.405000,5.153000


{'eval_loss': 0.2150246948003769,
 'eval_accuracy': 0.22123893805309736,
 'eval_f1': 0.5642547312919142,
 'eval_precision': 0.6489759150057459,
 'eval_recall': 0.5093704753694873,
 'eval_hamming_loss': 0.08711897738446411,
 'eval_runtime': 20.5692,
 'eval_samples_per_second': 82.405,
 'eval_steps_per_second': 5.153}

In [21]:
dev_outputs_roberta = trainer_roberta.predict(dev_dataset)

dev_logits_roberta = dev_outputs_roberta.predictions
dev_probs_roberta = 1 / (1 + np.exp(-dev_logits_roberta))

test_outputs_roberta = trainer_roberta.predict(test_dataset)

test_logits_roberta = test_outputs_roberta.predictions
test_probs_roberta = 1 / (1 + np.exp(-test_logits_roberta))

print("dev_probs_roberta:", dev_probs_roberta.shape)
print("test_probs_roberta:", test_probs_roberta.shape)

/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/daniel.ibanez/Desktop/uni/iln/proyecto/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


dev_probs_roberta: (1695, 18)
test_probs_roberta: (942, 18)


In [22]:
best_thresholds_roberta, best_f1s_roberta = find_best_thresholds(
    y_dev,
    dev_probs_roberta,
    genres
)

y_pred_roberta_thr = (dev_probs_roberta >= best_thresholds_roberta).astype(int)

metrics_roberta_thr = compute_metrics_debug(y_dev, y_pred_roberta_thr)
display(metrics_roberta_thr)

thresholds_roberta_df = pd.DataFrame({
    "genre": genres,
    "threshold": best_thresholds_roberta,
    "dev_f1_class": best_f1s_roberta,
    "support_dev": y_dev.sum(axis=0)
}).sort_values("genre")

display(thresholds_roberta_df)

{'accuracy': 0.20943952802359883,
 'f1': 0.6423708260894564,
 'precision': 0.6214935760687103,
 'recall': 0.6712011577325957,
 'hamming_loss': 0.09056047197640119,
 'avg_labels_pred': np.float64(2.8991150442477878),
 'avg_labels_true': np.float64(2.6601769911504425)}

,genre,threshold,dev_f1_class,support_dev
0,Action,0.51,0.761062,411
1,Adventure,0.29,0.649068,278
2,Animation,0.29,0.685879,172
3,Comedy,0.27,0.761905,625
4,Crime,0.39,0.671533,273
5,Drama,0.39,0.753927,789
6,Family,0.24,0.698413,178
7,Fantasy,0.28,0.580488,189
8,History,0.25,0.557522,90
9,Horror,0.26,0.710472,231


In [23]:
classic_model = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            analyzer="word",
            ngram_range=(1, 3),
            min_df=2,
            max_df=0.9,
            max_features=50000,
            sublinear_tf=True,
            stop_words="english"
        )),
        ("char_tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=50000,
            sublinear_tf=True
        ))
    ])),
    ("classifier", OneVsRestClassifier(
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            C=2.0,
            solver="liblinear"
        )
    ))
])

classic_model.fit(X_train, y_train)

dev_probs_classic = classic_model.predict_proba(X_dev)
test_probs_classic = classic_model.predict_proba(X_test)

print("dev_probs_classic:", dev_probs_classic.shape)
print("test_probs_classic:", test_probs_classic.shape)

dev_probs_classic: (1695, 18)
test_probs_classic: (942, 18)


In [24]:
best_thresholds_classic, best_f1s_classic = find_best_thresholds(
    y_dev,
    dev_probs_classic,
    genres
)

y_pred_classic_thr = (dev_probs_classic >= best_thresholds_classic).astype(int)

metrics_classic_thr = compute_metrics_debug(y_dev, y_pred_classic_thr)
display(metrics_classic_thr)

{'accuracy': 0.11209439528023599,
 'f1': 0.5946098964540322,
 'precision': 0.5598823256732718,
 'recall': 0.6521522821730397,
 'hamming_loss': 0.1149131432317273,
 'avg_labels_pred': np.float64(3.3079646017699114),
 'avg_labels_true': np.float64(2.6601769911504425)}

In [25]:
ensemble_results = []

for alpha in [0.5, 0.6, 0.7, 0.8, 0.9]:
    dev_probs_ensemble = alpha * dev_probs_roberta + (1 - alpha) * dev_probs_classic

    thresholds_ensemble, f1s_ensemble = find_best_thresholds(
        y_dev,
        dev_probs_ensemble,
        genres
    )

    y_pred_ensemble = (dev_probs_ensemble >= thresholds_ensemble).astype(int)

    metrics = compute_metrics_debug(y_dev, y_pred_ensemble)
    metrics["alpha_roberta"] = alpha
    ensemble_results.append(metrics)

ensemble_results_df = pd.DataFrame(ensemble_results).sort_values("f1", ascending=False)
display(ensemble_results_df)

,accuracy,f1,precision,recall,hamming_loss,avg_labels_pred,avg_labels_true,alpha_roberta
1,0.214749,0.672555,0.664393,0.692460,0.086857,2.899705,2.660177,0.6
2,0.215339,0.668817,0.659608,0.690601,0.086693,2.882596,2.660177,0.7
3,0.211209,0.666522,0.645630,0.697953,0.088037,2.938643,2.660177,0.8
0,0.206490,0.665958,0.651660,0.696883,0.089610,2.957522,2.660177,0.5
4,0.200590,0.657666,0.639946,0.689923,0.090331,2.977581,2.660177,0.9


In [26]:
ensemble_results_fine = []

for alpha in np.arange(0.50, 0.76, 0.025):
    dev_probs_ensemble = alpha * dev_probs_roberta + (1 - alpha) * dev_probs_classic

    thresholds_ensemble, f1s_ensemble = find_best_thresholds(
        y_dev,
        dev_probs_ensemble,
        genres
    )

    y_pred_ensemble = (dev_probs_ensemble >= thresholds_ensemble).astype(int)

    metrics = compute_metrics_debug(y_dev, y_pred_ensemble)
    metrics["alpha_roberta"] = alpha
    ensemble_results_fine.append(metrics)

ensemble_results_fine_df = pd.DataFrame(ensemble_results_fine).sort_values("f1", ascending=False)
display(ensemble_results_fine_df)

,accuracy,f1,precision,recall,hamming_loss,avg_labels_pred,avg_labels_true,alpha_roberta
4,0.214749,0.672555,0.664393,0.692460,0.086857,2.899705,2.660177,0.600
3,0.209440,0.671490,0.661142,0.694247,0.087283,2.919174,2.660177,0.575
5,0.208850,0.670413,0.658656,0.694131,0.087938,2.960472,2.660177,0.625
6,0.212389,0.670354,0.660355,0.690875,0.087414,2.930973,2.660177,0.650
2,0.212979,0.669810,0.659352,0.694957,0.088037,2.936283,2.660177,0.550
9,0.220059,0.669411,0.652547,0.697125,0.087348,2.917994,2.660177,0.725
7,0.218879,0.669074,0.659439,0.691912,0.086890,2.888496,2.660177,0.675
8,0.215339,0.668817,0.659608,0.690601,0.086693,2.882596,2.660177,0.700
10,0.208260,0.668671,0.640784,0.706227,0.088922,3.015929,2.660177,0.750
1,0.214749,0.668256,0.659848,0.690398,0.088102,2.906785,2.660177,0.525


In [27]:
BEST_ALPHA = 0.6

dev_probs_ensemble = BEST_ALPHA * dev_probs_roberta + (1 - BEST_ALPHA) * dev_probs_classic

best_thresholds_ensemble, best_f1s_ensemble = find_best_thresholds(
    y_dev,
    dev_probs_ensemble,
    genres
)

y_pred_ensemble_dev = (dev_probs_ensemble >= best_thresholds_ensemble).astype(int)

metrics_ensemble_dev = compute_metrics_debug(y_dev, y_pred_ensemble_dev)
display(metrics_ensemble_dev)

thresholds_ensemble_df = pd.DataFrame({
    "genre": genres,
    "threshold": best_thresholds_ensemble,
    "dev_f1_class": best_f1s_ensemble,
    "support_dev": y_dev.sum(axis=0)
}).sort_values("genre")

display(thresholds_ensemble_df)

{'accuracy': 0.21474926253687315,
 'f1': 0.6725546597747527,
 'precision': 0.6643931666862759,
 'recall': 0.6924596062327805,
 'hamming_loss': 0.08685676827269748,
 'avg_labels_pred': np.float64(2.8997050147492627),
 'avg_labels_true': np.float64(2.6601769911504425)}

,genre,threshold,dev_f1_class,support_dev
0,Action,0.47,0.774665,411
1,Adventure,0.33,0.672840,278
2,Animation,0.32,0.701950,172
3,Comedy,0.37,0.770801,625
4,Crime,0.34,0.685524,273
5,Drama,0.44,0.758621,789
6,Family,0.35,0.736264,178
7,Fantasy,0.37,0.592391,189
8,History,0.34,0.565657,90
9,Horror,0.40,0.709957,231


In [28]:
test_probs_ensemble = BEST_ALPHA * test_probs_roberta + (1 - BEST_ALPHA) * test_probs_classic

test_pred_bin_ensemble = (test_probs_ensemble >= best_thresholds_ensemble).astype(int)

for i in range(test_pred_bin_ensemble.shape[0]):
    if test_pred_bin_ensemble[i].sum() == 0:
        best_class = test_probs_ensemble[i].argmax()
        test_pred_bin_ensemble[i, best_class] = 1

test_pred_labels_ensemble = mlb.inverse_transform(test_pred_bin_ensemble)

test_pred_genres_ensemble = [
    ", ".join(labels)
    for labels in test_pred_labels_ensemble
]

In [29]:
submission_ensemble_df = pd.DataFrame({
    "movie_name": test_df["movie_name"],
    "description": test_df["description"],
    "genre": test_pred_genres_ensemble
})

display(submission_ensemble_df.head())
print(submission_ensemble_df.shape)

submission_ensemble_df.to_csv("dataset_test_preds.csv", index=False)

with zipfile.ZipFile("ILN05-Ensemble-TFIDF-RoBERTa.zip", "w") as zipf:
    zipf.write("dataset_test_preds.csv")

with zipfile.ZipFile("ILN05-Ensemble-TFIDF-RoBERTa.zip", "r") as zipf:
    print(zipf.namelist())

,movie_name,description,genre
0,Opposites Attract,"She's a divorce lawyer, single mother and perp...","Comedy, Drama, Romance"
1,A Turtle's Tale: Sammy's Adventures,A sea turtle who was hatched in 1959 spends th...,"Adventure, Animation, Comedy, Family"
2,My Stepmother Is an Alien,Trying to rescue her home planet from destruct...,"Drama, Romance, Science Fiction"
3,You've Got Mail,"Book superstore magnate, Joe Fox and independe...","Comedy, Drama, Romance"
4,The Thing,"In the winter of 1982, a twelve-man research t...","Horror, Science Fiction"


(942, 3)
['dataset_test_preds.csv']


In [30]:
check_df = pd.read_csv("dataset_test_preds.csv")

print(check_df.shape)
print(check_df.columns.tolist())
print(check_df["genre"].isna().sum())
print((check_df["genre"].str.len() == 0).sum())
display(check_df.head())

(942, 3)
['movie_name', 'description', 'genre']
0
0


,movie_name,description,genre
0,Opposites Attract,"She's a divorce lawyer, single mother and perp...","Comedy, Drama, Romance"
1,A Turtle's Tale: Sammy's Adventures,A sea turtle who was hatched in 1959 spends th...,"Adventure, Animation, Comedy, Family"
2,My Stepmother Is an Alien,Trying to rescue her home planet from destruct...,"Drama, Romance, Science Fiction"
3,You've Got Mail,"Book superstore magnate, Joe Fox and independe...","Comedy, Drama, Romance"
4,The Thing,"In the winter of 1982, a twelve-man research t...","Horror, Science Fiction"
